# Классификация SI > медиана



In [1]:
# Установка библиотек для Google Colab
!pip -q install xgboost lightgbm openpyxl seaborn


In [2]:
# Загрузка исходного файла с данными

import os
from google.colab import files

DATA_PATH = '/content/drug_data.xlsx'

if not os.path.exists(DATA_PATH):
    uploaded = files.upload()
    if 'drug_data.xlsx' not in uploaded:

        first_file = next(iter(uploaded.keys()))
        os.rename(first_file, DATA_PATH)

os.makedirs('reports', exist_ok=True)
os.makedirs('models', exist_ok=True)
print(f'Файл данных: {DATA_PATH}')
print('Папки reports/ и models/ готовы')


Файл данных: /content/drug_data.xlsx
Папки reports/ и models/ готовы


## Основной код

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
import warnings
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

class SIClassificationPredictor:
    def __init__(self):
        self.models = {}
        self.results = {}
        self.threshold = None

    def load_and_prepare_data(self):
        print("="*60)
        print("КЛАССИФИКАЦИЯ: SI > МЕДИАНА")
        print("="*60)

        df = pd.read_excel(DATA_PATH)
        df = df.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})

        if 'Unnamed: 0' in df.columns:
            df = df.drop('Unnamed: 0', axis=1)

        self.threshold = df['SI'].median()
        print(f"Медиана SI: {self.threshold:.3f}")

        y = (df['SI'] > self.threshold).astype(int)
        print(f"Класс 0: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
        print(f"Класс 1: {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

        feature_cols = [col for col in df.columns if col not in ['IC50', 'CC50', 'SI']]
        X = df[feature_cols].copy()

        # Очистка данных
        # Drop constant features
        constant_features = X.columns[X.nunique() <= 1]
        if len(constant_features) > 0:
            X = X.drop(constant_features, axis=1)
            print(f"Удалены константные признаки: {list(constant_features)}")

        # Drop columns that are entirely NaN
        all_nan_cols = X.columns[X.isnull().all()]
        if len(all_nan_cols) > 0:
            X = X.drop(columns=all_nan_cols)
            print(f"Удалены признаки, состоящие полностью из NaN: {list(all_nan_cols)}")

        # Impute remaining NaNs (partial NaNs) before splitting
        imputer = SimpleImputer(strategy='mean')
        X_imputed = imputer.fit_transform(X)
        X = pd.DataFrame(X_imputed, columns=X.columns, index=X.index)
        print(f"Количество NaN после импутации: {X.isnull().sum().sum()}")

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        print(f"Итоговое количество признаков: {X.shape[1]}")
        return X, y

    def define_models(self):
        self.models = {
            'Logistic Regression': Pipeline([
                ('scaler', StandardScaler()), # Imputer removed, as NaNs are handled globally
                ('model', LogisticRegression(random_state=42, max_iter=1000))
            ]),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            'XGBoost': xgb.XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
            'LightGBM': lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1)
        }

    def evaluate_and_save(self):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        final_results = {}

        for name, model in self.models.items():
            # Кросс-валидация
            scores = cross_val_score(model, self.X_train, self.y_train, cv=cv, scoring='f1')
            print(f"{name}: F1 = {scores.mean():.3f} ± {scores.std():.3f}")

            # Обучение и тестирование
            model.fit(self.X_train, self.y_train)
            y_pred = model.predict(self.X_test)
            y_pred_proba = model.predict_proba(self.X_test)[:, 1]

            final_results[name] = {
                'accuracy': accuracy_score(self.y_test, y_pred),
                'f1': f1_score(self.y_test, y_pred),
                'auc': roc_auc_score(self.y_test, y_pred_proba),
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba
            }

        # Сохранение результатов
        results_summary = []
        for name, metrics in final_results.items():
            results_summary.append({
                'Model': name,
                'Accuracy': metrics['accuracy'],
                'F1_Score': metrics['f1'],
                'ROC_AUC': metrics['auc']
            })

        results_df = pd.DataFrame(results_summary)
        results_df.to_csv('reports/si_median_classification_results.csv', index=False)

        # Лучшая модель
        best_model_name = max(final_results.keys(), key=lambda x: final_results[x]['f1'])
        print(f"\nЛучшая модель: {best_model_name}")
        print(f"F1-Score: {final_results[best_model_name]['f1']:.3f}")

        return final_results, best_model_name

def main():
    predictor = SIClassificationPredictor()
    X, y = predictor.load_and_prepare_data()
    predictor.define_models()
    final_results, best_model_name = predictor.evaluate_and_save()

if __name__ == "__main__":
    main()

КЛАССИФИКАЦИЯ: SI > МЕДИАНА
Медиана SI: 3.846
Класс 0: 501 (50.0%)
Класс 1: 500 (50.0%)
Удалены константные признаки: ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_N_O', 'fr_SH', 'fr_azide', 'fr_barbitur', 'fr_benzodiazepine', 'fr_diazo', 'fr_dihydropyridine', 'fr_isocyan', 'fr_isothiocyan', 'fr_lactam', 'fr_nitroso', 'fr_phos_acid', 'fr_phos_ester', 'fr_prisulfonamd', 'fr_thiocyan']
Количество NaN после импутации: 0
Итоговое количество признаков: 192
Logistic Regression: F1 = 0.647 ± 0.022
Random Forest: F1 = 0.667 ± 0.041
XGBoost: F1 = 0.649 ± 0.018
LightGBM: F1 = 0.665 ± 0.023

Лучшая модель: XGBoost
F1-Score: 0.636
